# 04 · Bronze – Auto Loader (raw → bronze)
Incrementally loads **new files only** from `raw/<sourcefolder>` into a Delta table at `bronze/<targetfolder>`.

- Used by both workflows: looped over the lookup array in Workflow 1, called once for `netflix_titles` in Workflow 2.
- `availableNow` trigger: processes everything new, then **stops**, so the task finishes.
- Adds lineage columns `_source_file` and `_ingested_at`.
- Checkpoint and schema location sit in `bronze/_checkpoints` and `bronze/_schemas`.

In [0]:
# 1. Use clean empty strings for blank widget defaults
dbutils.widgets.text("storage_account", "mynetflixproject")
dbutils.widgets.text("sourcefolder", "")
dbutils.widgets.text("targetfolder", "")

# 2. Strip accidental leading/trailing spaces when fetching values
ACCOUNT = dbutils.widgets.get("storage_account").strip()
SRC     = dbutils.widgets.get("sourcefolder").strip()
TGT     = dbutils.widgets.get("targetfolder").strip()

# 3. Guardrail: Fail fast if required widgets are empty to protect root containers
if not ACCOUNT or not SRC or not TGT:
    raise ValueError("Widgets 'storage_account', 'sourcefolder', and 'targetfolder' must all be non-empty.")

# 4. Safely construct the ADLS Gen2 path without double slashes or empty segments
def lake(container: str, *parts: str) -> str:
    clean_parts = [p.strip("/") for p in parts if p and p.strip("/")]
    base_url = f"abfss://{container.strip()}@{ACCOUNT}.dfs.core.windows.net"
    return f"{base_url}/{'/'.join(clean_parts)}" if clean_parts else base_url



In [0]:
from pyspark.sql import functions as F

raw_path        = lake("raw", SRC)
bronze_path     = lake("bronze", TGT)
checkpoint_path = lake("bronze", "_checkpoints", TGT)
schema_path     = lake("bronze", "_schemas", TGT)

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", True)
      .option("multiLine", True)      # quoted text may contain line breaks
      .option("escape", '"')           # a doubled quote ("") inside quoted text is a literal quote
      .option("cloudFiles.inferColumnTypes", True)
      .option("cloudFiles.schemaLocation", schema_path)
      .load(raw_path)
      .withColumn("_source_file", F.col("_metadata.file_path"))
      .withColumn("_ingested_at", F.current_timestamp()))

In [0]:
# Safely check if the raw folder exists before running Auto Loader
try:
    n_files = len([f for f in dbutils.fs.ls(raw_path) if not f.name.startswith("_")])
except Exception:
    print(f"[bronze] {SRC}: raw folder does not exist yet at {raw_path}. Skipping.")
    dbutils.notebook.exit(f"Skipped: {SRC} folder missing in raw")

print(f"[bronze] {SRC}: {n_files} file(s) found in raw")

query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .option("mergeSchema", True)
         .trigger(availableNow=True)
         .start(bronze_path))

query.awaitTermination()   # wait, so the workflow task reflects real success or failure

progress = query.lastProgress
rows = progress["numInputRows"] if progress else 0
print(f"[bronze] {SRC} -> {TGT}: done, {rows} new row(s) written this run")
if rows == 0:
    print("[bronze] nothing new: files already processed (checkpoint) or raw folder unchanged")
